# 🏢 Sistema de Gestión de Recursos Humanos
## Módulo 0 — Gestión de Departamentos
---
Este módulo permite **añadir**, **modificar** y **eliminar** departamentos de la empresa.

> **Base de datos:** `rrhh_sistema` · **Puerto:** `1989` · **Motor:** MySQL

## ⚙️ Celda 1 — Configuración e Importaciones
Ejecuta esta celda primero. Instala las dependencias si no las tienes.

In [ ]:
# ── Instalación automática si falta el conector ──────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'mysql-connector-python', 'ipywidgets', '--quiet'], check=False)

# ── Importaciones ─────────────────────────────────────────────────────────
import mysql.connector
from mysql.connector import Error
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

print('✅ Librerías cargadas correctamente.')

## 🔌 Celda 2 — Conexión a la Base de Datos
Establece la conexión con MySQL. Si hay un error, aparecerá el mensaje de fallo.

In [ ]:
# Conexion a la base de datos
import os
import mysql.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '1989')),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'rrhh_sistema'),
    'charset': 'utf8mb4',
}

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

try:
    conn = get_connection()
    conn.close()
    print(f'Conexion exitosa -> {DB_CONFIG["database"]} (Puerto {DB_CONFIG["port"]})')
except mysql.connector.Error as e:
    print(f'Error de conexion: {e}')



## 📋 Celda 3 — Funciones CRUD de Departamentos
Aquí se definen las operaciones: **Crear**, **Leer**, **Actualizar** y **Eliminar**.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              FUNCIONES CRUD — departamentos                  ║
# ╚══════════════════════════════════════════════════════════════╝

# ── CREATE ────────────────────────────────────────────────────────────────
def agregar_departamento(nombre: str, descripcion: str = '') -> str:
    """
    Inserta un nuevo departamento.
    Retorna un mensaje de éxito o error.
    """
    nombre = nombre.strip()
    if not nombre:
        return '⚠️  El nombre del departamento no puede estar vacío.'
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO departamentos (nombre_departamento, descripcion) VALUES (%s, %s)',
            (nombre, descripcion.strip() or None)
        )
        conn.commit()
        nuevo_id = cursor.lastrowid
        cursor.close(); conn.close()
        return f'✅ Departamento "{nombre}" creado con ID {nuevo_id}.'
    except Error as e:
        return f'❌ Error al crear: {e}'


# ── READ ──────────────────────────────────────────────────────────────────
def listar_departamentos() -> pd.DataFrame:
    """
    Devuelve todos los departamentos como DataFrame.
    """
    try:
        conn = get_connection()
        df = pd.read_sql(
            'SELECT id_departamento AS ID, nombre_departamento AS Nombre, '
            'COALESCE(descripcion, "—") AS Descripción '
            'FROM departamentos ORDER BY id_departamento',
            conn
        )
        conn.close()
        return df
    except Error as e:
        print(f'❌ Error al listar: {e}')
        return pd.DataFrame()


# ── UPDATE ────────────────────────────────────────────────────────────────
def modificar_departamento(id_dep: int, nuevo_nombre: str, nueva_desc: str = '') -> str:
    """
    Actualiza nombre y descripción del departamento con el ID dado.
    """
    nuevo_nombre = nuevo_nombre.strip()
    if not nuevo_nombre:
        return '⚠️  El nombre no puede quedar vacío.'
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute(
            'UPDATE departamentos SET nombre_departamento=%s, descripcion=%s '
            'WHERE id_departamento=%s',
            (nuevo_nombre, nueva_desc.strip() or None, id_dep)
        )
        conn.commit()
        filas = cursor.rowcount
        cursor.close(); conn.close()
        if filas == 0:
            return f'⚠️  No existe ningún departamento con ID {id_dep}.'
        return f'✅ Departamento ID {id_dep} actualizado a "{nuevo_nombre}".'
    except Error as e:
        return f'❌ Error al modificar: {e}'


# ── DELETE ────────────────────────────────────────────────────────────────
def eliminar_departamento(id_dep: int) -> str:
    """
    Elimina el departamento SOLO si no tiene empleados asignados.
    """
    try:
        conn = get_connection()
        cursor = conn.cursor()
        # Verificar que no haya empleados usando este departamento
        cursor.execute(
            'SELECT COUNT(*) FROM personal WHERE id_departamento = %s', (id_dep,)
        )
        (total,) = cursor.fetchone()
        if total > 0:
            cursor.close(); conn.close()
            return (f'⛔ No se puede eliminar: {total} empleado(s) '
                    f'pertenecen a este departamento.')
        cursor.execute(
            'DELETE FROM departamentos WHERE id_departamento = %s', (id_dep,)
        )
        conn.commit()
        filas = cursor.rowcount
        cursor.close(); conn.close()
        if filas == 0:
            return f'⚠️  No existe ningún departamento con ID {id_dep}.'
        return f'🗑️  Departamento ID {id_dep} eliminado correctamente.'
    except Error as e:
        return f'❌ Error al eliminar: {e}'

print('✅ Funciones CRUD cargadas.')

## 🖥️ Celda 4 — Interfaz Interactiva
Formulario con pestañas para cada operación. **Ejecuta esta celda para abrir el panel.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               INTERFAZ — ipywidgets                          ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Estilo común ──────────────────────────────────────────────────────────
STYLE   = {'description_width': '130px'}
LAYOUT  = widgets.Layout(width='420px')
BTN_LY  = widgets.Layout(width='160px', margin='10px 0 0 0')
OUT_LY  = widgets.Layout(border='1px solid #ccc', padding='8px',
                          min_height='40px', margin_top='6px')

# ══════════════════════════════════════════════════════════════════════════
# TAB 1 — VER DEPARTAMENTOS
# ══════════════════════════════════════════════════════════════════════════
out_ver   = widgets.Output(layout=OUT_LY)
btn_ver   = widgets.Button(description='🔄 Actualizar tabla',
                            button_style='info', layout=BTN_LY)

def on_ver(_):
    with out_ver:
        clear_output(wait=True)
        df = listar_departamentos()
        if df.empty:
            print('ℹ️  No hay departamentos registrados aún.')
        else:
            display(df.style.set_properties(**{'text-align': 'left'})
                           .set_table_styles(
                               [{'selector': 'th',
                                 'props': [('background-color', '#2c3e50'),
                                           ('color', 'white'),
                                           ('padding', '6px 12px')]}]))

btn_ver.on_click(on_ver)
tab1 = widgets.VBox([btn_ver, out_ver])


# ══════════════════════════════════════════════════════════════════════════
# TAB 2 — AÑADIR DEPARTAMENTO
# ══════════════════════════════════════════════════════════════════════════
txt_add_nombre = widgets.Text(description='Nombre:', style=STYLE, layout=LAYOUT,
                               placeholder='Ej: Tecnología')
txt_add_desc   = widgets.Text(description='Descripción:', style=STYLE, layout=LAYOUT,
                               placeholder='Opcional')
btn_add        = widgets.Button(description='➕ Agregar',
                                 button_style='success', layout=BTN_LY)
out_add        = widgets.Output(layout=OUT_LY)

def on_add(_):
    with out_add:
        clear_output(wait=True)
        msg = agregar_departamento(txt_add_nombre.value, txt_add_desc.value)
        print(msg)
        if msg.startswith('✅'):
            txt_add_nombre.value = ''
            txt_add_desc.value   = ''

btn_add.on_click(on_add)
tab2 = widgets.VBox([
    widgets.HTML('<b>Nuevo departamento</b>'),
    txt_add_nombre, txt_add_desc, btn_add, out_add
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 3 — MODIFICAR DEPARTAMENTO
# ══════════════════════════════════════════════════════════════════════════
int_mod_id     = widgets.BoundedIntText(description='ID a editar:', style=STYLE,
                                        layout=LAYOUT, min=1, value=1)
txt_mod_nombre = widgets.Text(description='Nuevo nombre:', style=STYLE, layout=LAYOUT,
                               placeholder='Ej: Recursos Humanos')
txt_mod_desc   = widgets.Text(description='Nueva desc.:', style=STYLE, layout=LAYOUT,
                               placeholder='Opcional')
btn_buscar     = widgets.Button(description='🔍 Buscar ID',
                                 button_style='warning', layout=BTN_LY)
btn_mod        = widgets.Button(description='💾 Guardar cambios',
                                 button_style='primary', layout=BTN_LY)
out_mod        = widgets.Output(layout=OUT_LY)

def on_buscar(_):
    """Pre-carga los datos actuales del departamento en los campos."""
    with out_mod:
        clear_output(wait=True)
        df = listar_departamentos()
        row = df[df['ID'] == int_mod_id.value]
        if row.empty:
            print(f'⚠️  No existe ningún departamento con ID {int_mod_id.value}.')
        else:
            nombre_actual = row.iloc[0]['Nombre']
            desc_actual   = row.iloc[0]['Descripción']
            txt_mod_nombre.value = nombre_actual
            txt_mod_desc.value   = '' if desc_actual == '—' else desc_actual
            print(f'📌 Cargado: "{nombre_actual}" — edita los campos y presiona Guardar.')

def on_mod(_):
    with out_mod:
        clear_output(wait=True)
        msg = modificar_departamento(
            int_mod_id.value, txt_mod_nombre.value, txt_mod_desc.value
        )
        print(msg)

btn_buscar.on_click(on_buscar)
btn_mod.on_click(on_mod)
tab3 = widgets.VBox([
    widgets.HTML('<b>Modificar departamento existente</b>'),
    int_mod_id,
    widgets.HBox([btn_buscar, widgets.Label('← primero busca el ID')]),
    txt_mod_nombre, txt_mod_desc,
    btn_mod, out_mod
])


# ══════════════════════════════════════════════════════════════════════════
# TAB 4 — ELIMINAR DEPARTAMENTO
# ══════════════════════════════════════════════════════════════════════════
int_del_id   = widgets.BoundedIntText(description='ID a eliminar:', style=STYLE,
                                      layout=LAYOUT, min=1, value=1)
chk_confirma = widgets.Checkbox(description='Confirmo que deseo eliminar este departamento',
                                 value=False,
                                 layout=widgets.Layout(width='440px'))
btn_del      = widgets.Button(description='🗑️ Eliminar',
                               button_style='danger', layout=BTN_LY)
out_del      = widgets.Output(layout=OUT_LY)

def on_del(_):
    with out_del:
        clear_output(wait=True)
        if not chk_confirma.value:
            print('⚠️  Marca la casilla de confirmación antes de eliminar.')
            return
        msg = eliminar_departamento(int_del_id.value)
        print(msg)
        if msg.startswith('🗑️'):
            chk_confirma.value = False

btn_del.on_click(on_del)
tab4 = widgets.VBox([
    widgets.HTML('<b>Eliminar departamento</b><br>'
                 '<small style="color:gray">⚠️ Solo se puede eliminar si no '
                 'tiene empleados asignados.</small>'),
    int_del_id, chk_confirma, btn_del, out_del
])


# ══════════════════════════════════════════════════════════════════════════
# PANEL PRINCIPAL — pestañas
# ══════════════════════════════════════════════════════════════════════════
tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4])
tabs.set_title(0, '📋 Ver departamentos')
tabs.set_title(1, '➕ Añadir')
tabs.set_title(2, '✏️  Modificar')
tabs.set_title(3, '🗑️  Eliminar')

header = widgets.HTML(
    '<div style="background:#2c3e50;color:white;padding:12px 16px;'
    'border-radius:6px 6px 0 0;font-family:Arial;font-size:15px;">'
    '🏢 <b>Módulo 0 — Gestión de Departamentos</b> | '
    'Sistema RRHH · rrhh_sistema</div>'
)

panel = widgets.VBox([header, tabs])
display(panel)

# Carga automática de la tabla al abrir
on_ver(None)

---
## 🧪 Celda 5 — Pruebas rápidas (opcional)
Puedes ejecutar estas celdas directamente sin usar el panel visual.

In [ ]:
# ── Ejemplo: Agregar departamentos de prueba ──────────────────────────────
print(agregar_departamento('Recursos Humanos', 'Gestión del talento humano'))
print(agregar_departamento('Tecnología',       'Desarrollo y sistemas'))
print(agregar_departamento('Finanzas',         'Contabilidad y presupuesto'))
print(agregar_departamento('Ventas',           'Comercialización y clientes'))
print(agregar_departamento('Operaciones'))

In [ ]:
# ── Ejemplo: Ver todos los departamentos ─────────────────────────────────
display(listar_departamentos())

In [ ]:
# ── Ejemplo: Modificar el departamento con ID 1 ──────────────────────────
print(modificar_departamento(1, 'Recursos Humanos', 'Área estratégica de personas'))

In [ ]:
# ── Ejemplo: Eliminar departamento (solo si no tiene empleados) ───────────
print(eliminar_departamento(5))   # Cambia el ID según lo que tengas